In [1]:
# Librerías generales
#=========================
import json
from json import dumps
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, explode, from_json, col, current_date
from pyspark.sql.types import StringType, StructType, StructField, ArrayType
import datetime 
import requests
import urllib3
import matplotlib as mtl
from datetime import datetime,date
import pytz
import io
import glob
import os
import boto3

In [2]:
CATALOG_URI   = "http://nessie:19120/api/v1"   # Nessie Server URI
WAREHOUSE     = "s3a://warehouse/"              # Bucket en MinIO
STORAGE_URI   = "http://minio:9000"             # Nombre de servicio Docker (no IP)
AWS_ACCESS_KEY = 'admin'
AWS_SECRET_KEY  = 'password'

In [3]:
# Configuración de Minio
minio_client = boto3.client(
    's3',
    #endpoint_url='http://172.18.0.4:9000',  # Usar el nombre del servicio de Docker
    endpoint_url='http://minio:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='password',
    region_name='us-east-1'
)

In [4]:
from pyspark.sql import SparkSession

# Las extensiones de Iceberg/Nessie ya vienen en spark-defaults.conf
# Aquí solo sobreescribimos lo que depende de variables (credenciales y URIs)
spark = (
    SparkSession.builder
    .appName("spark_load_to_iceberg")
    .master("spark://spark-master:7077")

    # Driver host: nombre del servicio Jupyter en la red Docker
    # Necesario para que los workers puedan conectarse de vuelta al driver
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")

    # --- Recursos: 1 worker, 2 cores, 2 GB ---
    .config("spark.executor.instances", "1")
    .config("spark.executor.cores", "2")
    .config("spark.executor.memory", "2g")
    .config("spark.driver.memory", "1g")

    # --- Catálogo Nessie (URIs dinámicas) ---
    .config("spark.sql.catalog.nessie.uri",       CATALOG_URI)
    .config("spark.sql.catalog.nessie.warehouse",  WAREHOUSE)
    .config("spark.sql.catalog.nessie.s3.endpoint", STORAGE_URI)

    # --- Credenciales S3/MinIO ---
    .config("spark.hadoop.fs.s3a.endpoint",    STORAGE_URI)
    .config("spark.hadoop.fs.s3a.access.key",  AWS_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key",  AWS_SECRET_KEY)
    .config("spark.sql.catalog.nessie.s3.access-key-id",     AWS_ACCESS_KEY)
    .config("spark.sql.catalog.nessie.s3.secret-access-key", AWS_SECRET_KEY)

    .getOrCreate()
)

print(f"Spark version : {spark.version}")
print(f"Master        : {spark.sparkContext.master}")

Spark version : 3.5.0
Master        : spark://spark-master:7077


In [5]:
# Celda eliminada — SparkSession ya se crea en la celda anterior

In [6]:
df_taxis = spark.read.parquet("s3a://bronze/yellow_tripdata_2025-01.parquet")


Py4JJavaError: An error occurred while calling o54.parquet.
: java.lang.RuntimeException: java.lang.ClassNotFoundException: Class org.apache.hadoop.fs.s3a.S3AFileSystem not found
	at org.apache.hadoop.conf.Configuration.getClass(Configuration.java:2688)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3431)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3466)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:174)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3574)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3521)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:540)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:365)
	at org.apache.spark.sql.execution.datasources.DataSource$.$anonfun$checkAndGlobPathIfNecessary$1(DataSource.scala:724)
	at scala.collection.immutable.List.map(List.scala:293)
	at org.apache.spark.sql.execution.datasources.DataSource$.checkAndGlobPathIfNecessary

In [ ]:
df_taxis.printSchema()



In [ ]:
df_taxis.show(5)

In [ ]:
display(df_taxis)

In [ ]:
pdf = df_taxis.limit(10).toPandas()
display(pdf)  

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.gold;").show()

In [ ]:
df_taxis.writeTo("nessie.gold.yellowtrip").createOrReplace()


In [ ]:
#spark.stop()